# 6 — Fleet-Wide Monitoring and Operations Report

**Sensor Intelligence Platform** — analytical walkthrough (6 / 7)

The earlier notebooks each isolated one capability on one signal. Operations work is the opposite: **every channel, every monitor, one ranked report**. Here we run forecasting health, anomaly detection, and drift across the full eight-sensor fleet during a simulated incident, then fold every finding into a single operator-facing Markdown report.

1. Simulate the fleet with a multi-channel incident.
2. Score forecasting health per sensor (held-out backtest).
3. Detect point-anomaly spikes fleet-wide (deseasonalised).
4. Scan for distributional drift per sensor.
5. Fold spikes and drift into alerts and render the ops report.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

NAVY, ORANGE, TEAL, RED, GREY = "#1f3a5f", "#e8893b", "#2a9d8f", "#c0392b", "#9aa0ad"

## 6.1 Simulate the fleet with an incident

We sample all eight channels for ten days and inject a coordinated incident: a temperature spike, a vibration drift, a pressure spike, and a slow motor-current drift. The rest of the fleet stays healthy, so the monitors must separate signal from background.

In [2]:
from sensor_intelligence.simulation import (
    SensorSimulator, SimulationConfig, AnomalyInjection, default_fleet,
)

PERIOD = 96
fleet = default_fleet()
units = {s.sensor_id: s.unit for s in fleet}
order = [s.sensor_id for s in fleet]
config = SimulationConfig(
    sensors=fleet, n_steps=PERIOD * 10, step_seconds=900, seed=29,
    anomalies=[
        AnomalyInjection('temperature', start_step=PERIOD * 6 + 30, magnitude=19.0),
        AnomalyInjection('vibration', start_step=PERIOD * 7, duration=PERIOD * 2,
                         magnitude=0.30, kind='drift'),
        AnomalyInjection('pressure', start_step=PERIOD * 8 + 12, magnitude=7.5),
        AnomalyInjection('motor_current', start_step=PERIOD * 5, duration=PERIOD * 3,
                         magnitude=5.0, kind='drift'),
    ],
)
df = SensorSimulator(config).run()
print(f'{df.shape[0]} rows, {df.sensor_id.nunique()} sensors, '
      f'{int(df.is_anomaly.sum())} labelled fault samples')

7680 rows, 8 sensors, 480 labelled fault samples


## 6.2 Fleet overview

A small-multiples grid is the operator's first glance: which channels are moving, and where. Injected faults are marked in red.

In [3]:
fig, axes = plt.subplots(4, 2, figsize=(13, 11), sharex=True)
for ax, sid in zip(axes.ravel(), order):
    s = df[df.sensor_id == sid]
    ax.plot(s.timestamp, s.value, color=NAVY, lw=0.7)
    faults = s[s.is_anomaly]
    if len(faults):
        ax.scatter(faults.timestamp, faults.value, color=RED, s=10, zorder=5)
    ax.set_title(f'{sid}  ({units[sid]})', fontsize=11)
fig.suptitle('Fleet overview — injected incident in red', y=0.995,
             fontsize=14, fontweight='bold')
fig.autofmt_xdate(); fig.tight_layout()

## 6.3 Forecasting health per sensor

A held-out backtest (the platform's `backtest` holds out the final day) of the tabular model on each channel shows where forecasts are reliable. Temperature — strong, stable seasonality — forecasts tightly with well-calibrated intervals; channels under an active drift (vibration, motor_current) or with a high noise-to-signal ratio show larger error and intervals that under-cover. That map of trust matters before acting on any forecast-based alert.

In [4]:
from sensor_intelligence.domain import TimeSeriesWindow
from sensor_intelligence.models import TabularForecaster
from sensor_intelligence.tracking import backtest

def window_for(sid):
    s = df[df.sensor_id == sid].sort_values('timestamp')
    return TimeSeriesWindow(sensor_id=sid, timestamps=list(s.timestamp),
                            values=[float(v) for v in s.value])

rows = []
for sid in order:
    w = window_for(sid)
    fn = lambda w, h: TabularForecaster(n_lags=2 * PERIOD).fit(w).forecast(h)
    _, m = backtest(fn, w, horizon=PERIOD)
    rows.append({'sensor': sid, 'unit': units[sid], 'mae': m['mae'],
                 'rmse': m['rmse'], 'coverage': m['interval_coverage']})
health = pd.DataFrame(rows).set_index('sensor').round(3)
health

,unit,mae,rmse,coverage
sensor,,,,
temperature,C,0.779,0.987,0.990
vibration,g,0.262,0.265,0.000
pressure,kPa,0.462,0.570,0.938
humidity,%RH,1.314,1.677,0.625
flow_rate,m3/h,2.496,2.962,0.656
motor_current,A,2.072,2.393,0.177
supply_voltage,V,1.359,1.729,0.448
shaft_speed,rpm,0.896,1.154,0.677


## 6.4 Fleet-wide point-anomaly detection

Most channels carry a strong daily cycle, which inflates a rolling detector's spread and hides spikes. We first **deseasonalise** each channel — subtract its time-of-day profile, estimated from the first three (fault-free) days — then run the robust z-score detector on the residual. Sustained **drifts** are deliberately left to the PSI scan in the next section; here we want the sharp spikes.

In [5]:
from sensor_intelligence.anomaly import RobustZScoreDetector

def deseasonalize(sid):
    s = df[df.sensor_id == sid].sort_values('timestamp').reset_index(drop=True)
    minute = s.timestamp.dt.hour * 60 + s.timestamp.dt.minute
    ref = s.iloc[:PERIOD * 3]                       # first three fault-free days
    profile = ref.groupby(ref.timestamp.dt.hour * 60
                          + ref.timestamp.dt.minute).value.mean()
    resid = s.value.to_numpy() - minute.map(profile).to_numpy()
    return list(s.timestamp), resid

detections = []
for sid in order:
    ts, resid = deseasonalize(sid)
    w = TimeSeriesWindow(sensor_id=sid, timestamps=ts, values=[float(v) for v in resid])
    detections += RobustZScoreDetector(window_size=PERIOD, threshold=8.0).detect(w)

by_sensor = (pd.Series([d.sensor_id for d in detections])
             .value_counts().reindex(order, fill_value=0))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(by_sensor.index, by_sensor.values, color=NAVY)
ax.set(title='Point-anomaly detections per sensor (deseasonalised)', ylabel='count')
ax.set_xticklabels(by_sensor.index, rotation=45, ha='right')
fig.tight_layout()
print(f'{len(detections)} point detections across the fleet')

2 point detections across the fleet


## 6.5 Drift scan

For each channel we compare an early reference window against the most recent window with the Population Stability Index. The channels carrying the slow drifts surface here even when no single point looks anomalous — the early-warning signal that a model's inputs have moved.

In [6]:
from sensor_intelligence.drift import PsiDriftDetector

ref_n, cur_n = PERIOD * 3, PERIOD * 2
drift_rows, drift_results = [], []
for sid in order:
    vals = df[df.sensor_id == sid].sort_values('timestamp').value.to_numpy()
    res = PsiDriftDetector(threshold=0.2).evaluate(vals[:ref_n], vals[-cur_n:])
    drift_results.append((sid, res))
    drift_rows.append({'sensor': sid, 'psi': round(res.score, 3), 'drifted': res.drifted})
drift = pd.DataFrame(drift_rows).set_index('sensor')
drift.sort_values('psi', ascending=False)

,psi,drifted
sensor,,
pressure,1.620,True
vibration,1.188,True
temperature,0.388,True
humidity,0.241,True
motor_current,0.240,True
supply_voltage,0.125,False
shaft_speed,0.060,False
flow_rate,0.053,False


## 6.6 Fold spikes and drift into alerts, then render the report

`AlertPolicy.from_anomalies` turns the point detections into severity-ranked alerts (merging any that share a sensor and timestamp); `AlertPolicy.from_drift` turns each drifted channel into a warning. `build_report` then renders the ranked alerts, a forecast summary, and the drift findings as Markdown an operator can read directly.

In [7]:
from sensor_intelligence.alerting import AlertPolicy
from sensor_intelligence.reporting import build_report, summarize_alerts

policy = AlertPolicy()
alerts = policy.from_anomalies(detections)
last_ts = df.timestamp.max()
for sid, res in drift_results:
    drift_alert = policy.from_drift(res, sensor_id=sid, timestamp=last_ts)
    if drift_alert is not None:
        alerts.append(drift_alert)
print('alert severity counts:', summarize_alerts(alerts))

# A representative forecast (temperature) and the drifted channels for the report.
temp_w = window_for('temperature')
train = TimeSeriesWindow(sensor_id='temperature',
                         timestamps=temp_w.timestamps[:-PERIOD],
                         values=temp_w.values[:-PERIOD])
forecast = TabularForecaster(n_lags=2 * PERIOD).fit(train).forecast(PERIOD)
drifted = [res for _, res in drift_results if res.drifted]

report = build_report('Pump-skid fleet — daily monitoring run', alerts,
                      forecast=forecast, drift=drifted, top_n=6)
print(report)

alert severity counts: {'critical': 2, 'warning': 5}


# Pump-skid fleet — daily monitoring run

## Alerts

- Total: 7
- Critical: 2
- Warning: 5

### Top 6 alerts

- **CRITICAL** `temperature` @ 2024-01-07T07:30:00 — CRITICAL: robust_zscore flagged sensor temperature (score 33.88)
    - modified z-score +33.88 exceeds 8
- **CRITICAL** `pressure` @ 2024-01-09T03:00:00 — CRITICAL: robust_zscore flagged sensor pressure (score 17.09)
    - modified z-score +17.09 exceeds 8
- **WARNING** `temperature` @ 2024-01-10T23:45:00 — WARNING: psi drift on sensor temperature (PSI 0.388 vs threshold 0.2)
- **WARNING** `vibration` @ 2024-01-10T23:45:00 — WARNING: psi drift on sensor vibration (PSI 1.188 vs threshold 0.2)
- **WARNING** `pressure` @ 2024-01-10T23:45:00 — WARNING: psi drift on sensor pressure (PSI 1.620 vs threshold 0.2)
- **WARNING** `humidity` @ 2024-01-10T23:45:00 — WARNING: psi drift on sensor humidity (PSI 0.241 vs threshold 0.2)

## Forecast

- Sensor: `temperature`
- Horizon: 96 steps (2024-01-10T00:00:00 → 2024-01-10T23:45:00)
- Mean

## 6.7 Severity at a glance

A stacked count of alerts by sensor and severity is the summary an on-call engineer triages from: where to look first, and how hot it is.

In [8]:
sev = pd.DataFrame([{'sensor': a.sensor_id, 'severity': a.severity.value} for a in alerts])
pivot = (sev.pivot_table(index='sensor', columns='severity', aggfunc='size', fill_value=0)
         if len(sev) else pd.DataFrame())
fig, ax = plt.subplots(figsize=(10, 4))
if len(pivot):
    colors = {'critical': RED, 'warning': ORANGE, 'info': GREY}
    bottom = np.zeros(len(pivot))
    for col in [c for c in ['critical', 'warning', 'info'] if c in pivot.columns]:
        ax.bar(pivot.index, pivot[col], bottom=bottom, label=col, color=colors[col])
        bottom += pivot[col].to_numpy()
    ax.legend(fontsize=9)
ax.set(title='Alerts by sensor and severity', ylabel='alerts')
ax.set_xticklabels(pivot.index, rotation=45, ha='right')
fig.tight_layout()

## Takeaways

- Fleet monitoring is the **integration test** for the platform: every channel, every monitor, one report.
- **Forecast health** varies by channel — strong seasonality forecasts tightly, near-stationary noise does not.
- Detectors are used **where each is strong**: a deseasonalised point detector for sharp spikes, the PSI scan for sustained drift.
- `AlertPolicy` + `build_report` turn raw detections and drift findings into a **ranked, explained, operator-ready** summary.
- To compare model choices behind these forecasts systematically, see experiment tracking in [notebook 7](07_experiment_tracking.ipynb).